# CNN Training - 16 Variasi Arsitektur

Notebook untuk melatih 16 variasi arsitektur CNN (shared parameters) dan 1 model
LocallyConnected2D (non-shared parameters).

Grid variasi (2×2×2×2 = **16**):
- Jumlah layer konvolusi: 2 atau 3
- Banyak filter per layer: base [32,64,128] atau large [64,128,256]
- Ukuran kernel: 3 atau 5
- Jenis pooling: max atau average

In [ ]:
import os, sys, time, pickle
import numpy as np
from pathlib import Path


def _find_root(marker="requirements.txt"):
    p = Path(os.getcwd())
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Repo root tidak ketemu, pastikan requirements.txt ada di root.")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
print("Working dir:", REPO_ROOT)

In [ ]:
sys.path.insert(0, str(REPO_ROOT / "src"))

import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

from cnn.train_keras import (
    build_conv2d_model, build_locally_connected_model,
    get_data_loaders, train, IMG_SIZE
)

tf.random.set_seed(42)
np.random.seed(42)

## Config

In [ ]:
MODELS_DIR  = "models/cnn"
HISTORY_DIR = "models/cnn/history"

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

## Load Data

In [ ]:
print("Loading data for standard models (150x150)...")
train_ds, val_ds = get_data_loaders(img_size=IMG_SIZE)

print("Loading data for LocallyConnected2D (64x64, smaller to save memory)...")
train_ds_lc, val_ds_lc = get_data_loaders(img_size=(64, 64))

## Definisi 16 Variasi Arsitektur

| Nama | N Layer | Filter | Kernel | Pooling |
|---|---|---|---|---|
| cnn-2L-base-k3-max | 2 | [32,64] | 3 | Max |
| cnn-2L-base-k3-avg | 2 | [32,64] | 3 | Avg |
| cnn-2L-base-k5-max | 2 | [32,64] | 5 | Max |
| cnn-2L-base-k5-avg | 2 | [32,64] | 5 | Avg |
| cnn-2L-large-k3-max | 2 | [64,128] | 3 | Max |
| cnn-2L-large-k3-avg | 2 | [64,128] | 3 | Avg |
| cnn-2L-large-k5-max | 2 | [64,128] | 5 | Max |
| cnn-2L-large-k5-avg | 2 | [64,128] | 5 | Avg |
| cnn-3L-base-k3-max | 3 | [32,64,128] | 3 | Max |
| cnn-3L-base-k3-avg | 3 | [32,64,128] | 3 | Avg |
| cnn-3L-base-k5-max | 3 | [32,64,128] | 5 | Max |
| cnn-3L-base-k5-avg | 3 | [32,64,128] | 5 | Avg |
| cnn-3L-large-k3-max | 3 | [64,128,256] | 3 | Max |
| cnn-3L-large-k3-avg | 3 | [64,128,256] | 3 | Avg |
| cnn-3L-large-k5-max | 3 | [64,128,256] | 5 | Max |
| cnn-3L-large-k5-avg | 3 | [64,128,256] | 5 | Avg |

In [ ]:
FILTER_CONFIGS = {
    "base":  {2: [32, 64],       3: [32, 64, 128]},
    "large": {2: [64, 128],      3: [64, 128, 256]},
}

VARIATIONS = []
for n_layers in [2, 3]:
    for f_type in ["base", "large"]:
        for k_size in [3, 5]:
            for p_type in ["max", "avg"]:
                name = f"cnn-{n_layers}L-{f_type}-k{k_size}-{p_type}"
                VARIATIONS.append({
                    "name": name,
                    "num_conv_layers": n_layers,
                    "filters": FILTER_CONFIGS[f_type][n_layers],
                    "kernel_sizes": [k_size] * n_layers,
                    "pooling": p_type,
                })

print(f"Total arsitektur: {len(VARIATIONS)}")
for v in VARIATIONS:
    print(f"  {v['name']:35s} | filters={v['filters']} | kernel={v['kernel_sizes'][0]} | pool={v['pooling']}")

## Training 16 Variasi Conv2D

In [ ]:
histories = {}

for v in VARIATIONS:
    name = v["name"]
    print(f"\n{'='*60}\nTraining: {name}\n{'='*60}")

    kwargs = {k: val for k, val in v.items() if k != "name"}
    model = build_conv2d_model(**kwargs)

    start_time = time.time()
    history = train(model, train_ds, val_ds, model_name=name)
    elapsed = time.time() - start_time

    print(f"Selesai dalam {int(elapsed)}s  |  best val_loss: {min(history.history['val_loss']):.4f}")

    histories[name] = history.history
    with open(os.path.join(HISTORY_DIR, f"{name}_history.pkl"), "wb") as f:
        pickle.dump(history.history, f)

# Simpan semua histories sekaligus
with open(os.path.join(MODELS_DIR, "all_conv2d_histories.pkl"), "wb") as f:
    pickle.dump(histories, f)

print("\nOK Semua 16 variasi Conv2D selesai dilatih.")

## Training Model LocallyConnected2D (Non-Shared)

In [ ]:
print(f"\n{'='*60}\nTraining: lc-model\n{'='*60}")
lc_model = build_locally_connected_model()

start_time = time.time()
lc_history = train(lc_model, train_ds_lc, val_ds_lc, model_name="lc-model")
elapsed = time.time() - start_time

print(f"Selesai dalam {int(elapsed)}s  |  best val_loss: {min(lc_history.history['val_loss']):.4f}")

with open(os.path.join(HISTORY_DIR, "lc-model_history.pkl"), "wb") as f:
    pickle.dump(lc_history.history, f)

print("\nOK Training LocallyConnected2D selesai.")